## Function

### Global Functions

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def parse_vcd(vcd_path):
    vcd_path = Path(vcd_path)
    text = vcd_path.read_text(errors="replace")
    match = re.search(r"\$timescale\s+(\d+)\s*(s|ms|us|ns|ps|fs)\s+\$end", text)
    if not match:
        raise ValueError(f"No VCD timescale found in {vcd_path}")
    scale = {"s": 1, "ms": 1e-3, "us": 1e-6, "ns": 1e-9, "ps": 1e-12, "fs": 1e-15}
    tick_to_ns = int(match.group(1)) * scale[match.group(2)] * 1e9
    code_by_name, width_by_name, histories = {}, {}, {}
    scopes, reading_values, tick = [], False, 0
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if not reading_values:
            if line.startswith("$scope"):
                scopes.append(line.split()[2])
            elif line.startswith("$upscope"):
                if scopes:
                    scopes.pop()
            elif line.startswith("$var"):
                fields = line.split()
                width, code, reference = int(fields[2]), fields[3], fields[4]
                code_by_name[".".join(scopes + [reference])] = code
                width_by_name[".".join(scopes + [reference])] = width
                histories.setdefault(code, [])
            elif line.startswith("$enddefinitions"):
                reading_values = True
            continue
        if line.startswith("#"):
            tick = int(line[1:])
        elif line.startswith("$"):
            continue
        elif line[0] in "bB":
            fields = line[1:].split()
            if len(fields) != 2:
                continue
            bits, code = fields
            value = None if any(bit in "xXzZ" for bit in bits) else int(bits, 2)
            if code in histories:
                histories[code].append((tick * tick_to_ns, value))
        elif line[0] in "01xXzZ":
            code = line[1:]
            bit = line[0].lower()
            value = int(bit) if bit in "01" else None
            if code in histories:
                histories[code].append((tick * tick_to_ns, value))
    max_time_ns = max((time for events in histories.values() for time, _ in events), default=0.0)
    return {
        "path": vcd_path,
        "code_by_name": code_by_name,
        "width_by_name": width_by_name,
        "histories": histories,
        "tick_to_ns": tick_to_ns,
        "max_time_ns": max_time_ns,
    }


def available_signals(data, scope=None, search=None, include_internal=True):
    rows = []
    for name, width in sorted(data["width_by_name"].items()):
        if scope and not (name == scope or name.startswith(f"{scope}.")):
            continue
        relative = name[len(scope) + 1:] if scope and name.startswith(f"{scope}.") else name
        if not include_internal and "." in relative:
            continue
        if search and search.casefold() not in relative.casefold():
            continue
        rows.append({"signal": name, "width_bits": width})
    return pd.DataFrame(rows, columns=["signal", "width_bits"])


def load_all_to_preview(vcd_path, *, scope=None, include_internal=True, start_ns=None, end_ns=None, show_values=False):
    data = parse_vcd(vcd_path)
    signals = available_signals(data, scope=scope, include_internal=include_internal)
    print(f"Loaded {len(data['code_by_name'])} VCD references from {data['path']} ({data['max_time_ns']:g} ns)")
    display(signals)
    figure, axes = plot_waveforms(
        data,
        signals["signal"].tolist(),
        start_ns=start_ns,
        end_ns=end_ns,
        show_values=show_values,
        title=f"All available signals: {scope or data['path'].stem}",
    )
    return data, signals, figure, axes


### General Functions

In [ ]:
def waveform(data, signal):
    if signal not in data["code_by_name"]:
        raise KeyError(f"Signal not found in VCD: {signal}")
    events = data["histories"][data["code_by_name"][signal]]
    times = np.asarray([time for time, _ in events], dtype=float)
    values = np.asarray([np.nan if value is None else value for _, value in events], dtype=float)
    return times, values


def visible_value_segments(data, signal, start_ns, end_ns):
    if end_ns <= start_ns:
        raise ValueError("end_ns must be greater than start_ns")
    events_by_time = {}
    for time_ns, value in data["histories"][data["code_by_name"][signal]]:
        events_by_time[time_ns] = value
    events = sorted(events_by_time.items())
    previous = None
    for time_ns, value in events:
        if time_ns <= start_ns:
            previous = value
    points = [(start_ns, previous)]
    points.extend((time_ns, value) for time_ns, value in events if start_ns < time_ns <= end_ns)
    compact = []
    for point in points:
        if not compact or point[1] != compact[-1][1]:
            compact.append(point)
    return [
        (left, compact[i + 1][0] if i + 1 < len(compact) else end_ns, value)
        for i, (left, value) in enumerate(compact)
        if (compact[i + 1][0] if i + 1 < len(compact) else end_ns) > left
    ]


def format_bus_value(data, signal, value, *, formats=None, enums=None):
    if value is None:
        return "X"
    if enums and signal in enums and value in enums[signal]:
        return str(enums[signal][value])
    options = (formats or {}).get(signal, {})
    width = data["width_by_name"][signal]
    number = int(value)
    if options.get("signed", False) and number & (1 << (width - 1)):
        number -= 1 << width
    radix = options.get("radix", "hex").lower()
    if radix in ("dec", "decimal"):
        return str(number)
    magnitude = abs(number)
    if radix == "bin":
        text = format(magnitude, "b")
        prefix = "0b"
    elif radix == "oct":
        text = format(magnitude, "o")
        prefix = "0o"
    elif radix == "hex":
        text = format(magnitude, "X")
        prefix = "0x"
    else:
        raise ValueError("radix must be 'bin', 'oct', 'dec', or 'hex'")
    if len(text) > 10:
        text = f"{text[:5]}…{text[-3:]}"
    return f"-{prefix}{text}" if number < 0 else f"{prefix}{text}"


def _show_bus_value(show_values, signal):
    if isinstance(show_values, dict):
        return show_values.get(signal, True)
    return bool(show_values)


def plot_waveforms(
    data,
    signals,
    *,
    labels=None,
    formats=None,
    enums=None,
    show_values=True,
    start_ns=None,
    end_ns=None,
    markers=(),
    shaded_intervals=(),
    shade_level=0.78,
    title="Selected VCD signals",
    output_path=None,
):
    signals = list(signals)
    if not signals:
        raise ValueError("Select at least one signal")
    missing = [name for name in signals if name not in data["code_by_name"]]
    if missing:
        raise KeyError(f"Signals not found in VCD: {missing}")
    if not 0.0 <= shade_level <= 1.0:
        raise ValueError("shade_level must be between 0.0 and 1.0")
    for interval in shaded_intervals or ():
        if len(interval) != 2 or float(interval[1]) <= float(interval[0]):
            raise ValueError(f"Invalid shaded interval: {interval}")

    start = 0.0 if start_ns is None else float(start_ns)
    end = data["max_time_ns"] + 1 if end_ns is None else float(end_ns)
    if end <= start:
        raise ValueError("end_ns must be greater than start_ns")
    height = max(3.2, 0.72 * len(signals) + 1.1)
    figure, axes = plt.subplots(
        len(signals), 1, figsize=(16, height), sharex=True,
        constrained_layout=False, squeeze=False, gridspec_kw={"hspace": 0},
    )
    figure.subplots_adjust(left=0.08, right=0.995, top=0.92, bottom=0.12, hspace=0)
    axes = axes[:, 0]

    for index, (axis, signal) in enumerate(zip(axes, signals)):
        width = data["width_by_name"][signal]
        name = labels[index] if labels else signal.split(".")[-1]
        segments = visible_value_segments(data, signal, start, end)
        if width == 1:
            for segment_index, (left, right, value) in enumerate(segments):
                if value is None:
                    continue
                axis.hlines(int(value), left, right, color="#1b7358", linewidth=1.45, zorder=2.0)
                if segment_index and segments[segment_index - 1][2] is not None and segments[segment_index - 1][2] != value:
                    axis.vlines(left, min(segments[segment_index - 1][2], value), max(segments[segment_index - 1][2], value), color="#1b7358", linewidth=1.45, zorder=2.0)
            axis.set_ylim(-0.2, 1.2)
            axis.set_yticks([0, 1])
        else:
            for left, right, value in segments:
                if value is None:
                    continue
                axis.broken_barh([(left, right - left)], (0.0, 1.0), facecolors="#eaf4ef", edgecolors="none", zorder=0.5)
                axis.broken_barh([(left, right - left)], (0.0, 1.0), facecolors="none", edgecolors="#1b7358", linewidth=1, zorder=2.5)
                if _show_bus_value(show_values, signal):
                    label = format_bus_value(data, signal, value, formats=formats, enums=enums)
                    axis.text((left + right) / 2, 0.5, label, ha="center", va="center", fontsize=6.5, color="#203a32", clip_on=True, zorder=3)
            axis.set_ylim(0, 1)
            axis.set_yticks([])
        axis.set_ylabel(name, rotation=0, ha="right", va="center", labelpad=12)
        axis.set_xlim(start, end)
        axis.margins(x=0)

    for interval in shaded_intervals or ():
        left, right = map(float, interval)
        left, right = max(left, start), min(right, end)
        if left < right:
            for axis in axes:
                axis.axvspan(left, right, color="#c9e2d6", alpha=shade_level, zorder=1.0)
    for marker in markers or ():
        marker = float(marker)
        if start <= marker <= end:
            for axis in axes:
                axis.axvline(marker, color="#2563eb", linewidth=1.15, linestyle="--", alpha=0.9, zorder=10)

    axes[0].set_title(title)
    axes[-1].set_xlabel("Simulation time (ns)")
    if output_path:
        figure.savefig(output_path, dpi=180, bbox_inches="tight")
    return figure, axes


### Specifics Functions

In [ ]:
def plot_named_signals(data, signals, **options):
    return plot_waveforms(data, signals, **options)


def plot_time_range(data, signals, start_ns, end_ns, **options):
    return plot_waveforms(data, signals, start_ns=start_ns, end_ns=end_ns, **options)


def plot_with_markers(data, signals, markers, **options):
    return plot_waveforms(data, signals, markers=markers, **options)


def plot_with_shading(data, signals, intervals, *, shade_level=0.78, **options):
    return plot_waveforms(data, signals, shaded_intervals=intervals, shade_level=shade_level, **options)


## Examples

### Load All to Preview First

In [ ]:
from pathlib import Path

VCD_PATH = Path("../axi_read_tb.vcd")
SCOPE = "tb_axi_read"
data, all_signals, all_figure, all_axes = load_all_to_preview(
    VCD_PATH,
    scope=SCOPE,
    include_internal=True,
    show_values=False,
)
plt.show()


### Load Specific Signal Names

In [ ]:
SELECTED_SIGNALS = [
    "tb_axi_read.start",
    "tb_axi_read.m_axi_arvalid",
    "tb_axi_read.m_axi_arready",
    "tb_axi_read.m_axi_rvalid",
    "tb_axi_read.m_axi_rready",
    "tb_axi_read.fifo_valid",
    "tb_axi_read.m_axi_araddr",
    "tb_axi_read.m_axi_rdata",
]
FORMATS = {
    "tb_axi_read.m_axi_araddr": {"radix": "hex", "signed": False},
    "tb_axi_read.m_axi_rdata": {"radix": "hex", "signed": False},
}
SHOW_VALUES = {
    "tb_axi_read.m_axi_araddr": False,
    "tb_axi_read.m_axi_rdata": False,
}
selected_figure, selected_axes = plot_named_signals(
    data,
    SELECTED_SIGNALS,
    formats=FORMATS,
    show_values=SHOW_VALUES,
    title="AXI read handshake and data",
)
plt.show()


### Load by Signal Time Range

In [ ]:
range_figure, range_axes = plot_time_range(
    data,
    SELECTED_SIGNALS,
    start_ns=80,
    end_ns=320,
    formats=FORMATS,
    show_values=SHOW_VALUES,
    title="AXI read signals: 80–320 ns",
)
plt.show()


### Marker

In [ ]:
marker_figure, marker_axes = plot_with_markers(
    data,
    SELECTED_SIGNALS,
    markers=[90, 195, 320],
    formats=FORMATS,
    show_values=SHOW_VALUES,
    title="AXI read signals with time markers",
)
plt.show()


### Shading in Simulation Time

In [ ]:
shade_figure, shade_axes = plot_with_shading(
    data,
    SELECTED_SIGNALS,
    intervals=[(90, 170), (240, 320)],
    shade_level=0.78,
    formats=FORMATS,
    show_values=SHOW_VALUES,
    title="AXI read signals with shaded intervals",
)
plt.show()
